In [2]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [12]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

APT High Voltage Amplifier initialized
Stage Moving Done
Stage disconnected


In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [14]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 1.4,
        "step_number": 40,
        "position_z": 0,
        "step_size_z": 0.5,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_1_Left",
        # chip_name="SiN_beam",
        # sample_name="w2_sweep",  # test_1_right, w=20
        sample_name="test_AFM4_450_w20_4_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 2.5%
process completed: 5.0%
process completed: 7.5%
process completed: 10.0%
process completed: 12.5%
process completed: 15.0%
process completed: 17.5%
process completed: 20.0%
process completed: 22.5%
process completed: 25.0%
process completed: 27.5%
process completed: 30.0%
process completed: 32.5%
process completed: 35.0%
process completed: 37.5%
process completed: 40.0%
process completed: 42.5%
process completed: 45.0%
process completed: 47.5%
process completed: 50.0%
process completed: 52.5%
process completed: 55.0%
process completed: 57.5%
process completed: 60.0%
process completed: 62.5%
process completed: 65.0%
process completed: 67.5%
process completed: 70.0%
process completed: 72.5%
process completed: 75.0%
process completed: 77.5%
process completed: 80.0%
process completed: 82.5%
process complet

({'position': [0.0,
   0.498672444837794,
   0.992461928159429,
   1.48686178167058,
   1.98492385631886,
   2.47566148869289,
   2.97433393353069,
   3.473616748558,
   3.95947141941588,
   4.45936460463271,
   4.93850520340587,
   5.43778801843318,
   5.93646046327097,
   6.43513290810877,
   6.92770165105136,
   7.42820520645772,
   7.92748802148503,
   8.42738120670186,
   8.92544328135014,
   9.42289498580889,
   9.92400891140477,
   10.4245124668111,
   10.9250160222175,
   11.4255195776238,
   11.9260231330302,
   12.4253059480575,
   12.9233680227058,
   13.4226508377331,
   13.923764763329,
   14.4212164677877,
   14.9217200231941,
   15.4228339487899,
   15.9233375041963,
   16.4159062471389,
   16.9164098025452,
   17.4169133579516,
   17.917416913358,
   18.4191412091433,
   18.9202551347392,
   19.4207586901456],
  'voltage': [[-0.07483476,
    -0.07553651,
    -0.07548246,
    -0.07507604,
    -0.07505155,
    -0.07563107,
    -0.07524815,
    -0.07495387,
    -0.07513279